Chargement et séparation des données 

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split 

file_path = "/Users/maximebenharrats/Downloads/blackSwan_data/blackSwan_data.parquet"
df = pd.read_parquet(file_path)

#extraction de X et y 
y = df["labels"].values

colonnes_a_retirer = ["labels", "detailed_labels", "weights"]
X = df.drop(columns=colonnes_a_retirer, errors='ignore').values

#extraction des poids 
weights = df["weights"].values

#séparation des données valides et d'entraînement
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(X, y, weights, random_state=1, stratify=y, test_size=0.2)

Définition de la fonction de calcul de la significance ; le seuil de détection est fixé ici 

In [2]:
import numpy as np 

def compute_significance(y_true, y_pred, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.full(len(y_true), 1.0)
    else:
        sample_weight = np.asarray(sample_weight)
        
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    selected = (y_pred >= 0.80) #seuil de détection à 80%

    s = np.sum(sample_weight[(y_true == 1) & selected])
    b = np.sum(sample_weight[(y_true == 0) & selected])

    if b == 0: b = 1.0
    if s <= 0 or b <= 0: return 0.0

    ams = np.sqrt(2 * ((s + b) * np.log(1 + s / b) - s))
    return float(ams)

Configuration du modèle HiggsNN 

In [ ]:
from sklearn.preprocessing import StandardScaler 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

class HiggsNN: 
    def __init__(self, input_dim, shared_layer_size, n_layers, activation, lr, dropout):
        self.scaler = StandardScaler()
        self.model = Sequential()

        #Première couche 
        self.model.add(tf.keras.Input(shape=(input_dim,)))

        #Couches cachées 
        for _ in range(n_layers):
            self.model.add(Dense(shared_layer_size, activation=activation))
            #on n'ajoute le dropout que si le taux est supérieur à 0 pour éviter de faire des calculs inutiles
            if dropout > 0:
                self.model.add(Dropout(dropout)) 
        
        #Couche de sortie 
        self.model.add(Dense(1, activation = 'sigmoid'))

        #définition de la méthode d'optimisation (descente de gradient, fonction de perte, et métriques d'évaluation)
        self.model.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])

### Entrainement du modèle ; on fixe epochs à 20 pour le test HPO rapide, mais on peut augmenter ce nombre pour de meilleurs résultats
    def fit(self, train_data, y_train, weights_train=None, X_val=None, y_val=None, weights_val=None, epochs=20):
        X_train_scaled = self.scaler.fit_transform(train_data)
        
        validation_data = None
        if X_val is not None and y_val is not None:
            X_val_scaled = self.scaler.transform(X_val)
            if weights_val is not None:
                validation_data = (X_val_scaled, y_val, weights_val)
            else:
                validation_data = (X_val_scaled, y_val)

        history = self.model.fit(
            X_train_scaled, y_train,
            sample_weight=weights_train,
            validation_data=validation_data,
            batch_size=2048,  # On augmente le batch_size car le GPU adore les gros blocs !
            epochs=epochs,
            verbose=0
        )
        return history 
    
    def predict(self, data):
        data_scaled = self.scaler.transform(data)
        return self.model.predict(data_scaled, batch_size=4096, verbose=0).flatten()


HPO avec Optuna 

In [ ]:
import optuna 

#on a déjà défini les données d'entrainement et de validation globalement, on n'a plus besoin de les charger à chaque essai d'Optuna

def objective(trial):
    # Définition de l'espace de recherche des hyperparamètres sur la base de l'article de recherche 
    shared_layer_size = trial.suggest_int('shared_layer_size', 16, 128)
    n_layers = trial.suggest_int('n_layers', 2, 6)
    activation = trial.suggest_categorical('activation', ['relu', 'swish', 'tanh'])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.0, 0.3)
    
    # Création et entraînement du modèle
    nn = HiggsNN(
        input_dim=X_train.shape[1],
        shared_layer_size=shared_layer_size,
        n_layers=n_layers,
        activation=activation,
        lr=lr,
        dropout=dropout
    )
    
    nn.fit(X_train, y_train, w_train, X_val, y_val, w_val, epochs=20) #on réduit le nombre d'epochs pour le test HPO rapide, mais on peut augmenter ce nombre pour de meilleurs résultats
    
    # Évaluation avec le score Z au seuil fixe de 0.80
    y_pred = nn.predict(X_val)
    score = compute_significance(y_val, y_pred, sample_weight=w_val)
    
    return score

# Lancement de l'étude Optuna 
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) #on fait 30 essais avec des hyperparamètres différents pour trouver les meilleurs, on peut augmenter ce nombre pour de meilleurs résultats

print("\n=== OPTIMISATION TERMINÉE ===")
print("Meilleurs paramètres trouvés :", study.best_params)
print("Meilleur significance atteint :", study.best_value)

A l'issue de ce code, on obtient dans le terminal : 

=== OPTIMISATION TERMINÉE ===
Meilleurs paramètres trouvés : {'shared_layer_size': 127, 'n_layers': 4, 'activation': 'swish', 'lr': 0.00017282986292292438, 'dropout': 0.0004620579301656755}
Meilleur significance atteint : 2.0319947286312026

On peut donc construire un NN avec les bons paramètres 

In [ ]:
nn_optimise = HiggsNN(
    input_dim=X_train.shape[1],
    shared_layer_size=study.best_params['shared_layer_size'],
    n_layers=study.best_params['n_layers'],
    activation=study.best_params['activation'],
    lr=study.best_params['lr'],
    dropout=study.best_params['dropout']
)

nn_optimise.fit(X_train, y_train, w_train, X_val, y_val, w_val, epochs=100) 
nn_optimise.predict(X_val)
print("Score final sur les données de validation :", compute_significance(y_val, nn_optimise.predict(X_val), sample_weight=w_val))